Рассморим новости для наших компаний

Импортируем библиотеки

In [ ]:
import requests
import json
from finhub_api_key import api_key
import pandas as pd
from datetime import datetime, timezone
import logging
from logger_config import logs, details

Мы получаем партнеров компании внутри сектора по тикеру

In [ ]:
if logs:
    try:
        logging.basicConfig(level=logging.INFO, filename="py_log.log",filemode="a",
                        format="%(asctime)s %(levelname)s %(message)s")


        def get_peers(ticker):

            params = {
                'token': api_key,
                'symbol': ticker,
                'grouping': 'sector'
            }

            response = requests.get(url='https://finnhub.io/api/v1/stock/peers', params=params)

            return response.json()
        logging.info(f"{filen}, success.")
    except Exception as e:
        logging.error(f"{filen}, error {e}")
else:
    def get_peers(ticker):

            params = {
                'token': api_key,
                'symbol': ticker,
                'grouping': 'sector'
            }

            response = requests.get(url='https://finnhub.io/api/v1/stock/peers', params=params)

            return response.json()

In [ ]:
get_peers('AAPL')

Пишем фукнцию получения новостей по тикеру 

In [ ]:
if logs:
    try:
        logging.basicConfig(level=logging.INFO, filename="py_log.log",filemode="a",
                        format="%(asctime)s %(levelname)s %(message)s")
        def get_news(ticker, dfrom,to):

            params = {
                'token': api_key,
                'symbol': ticker,
                'from': dfrom,
                'to': to
            }

            response = requests.get(url='https://finnhub.io/api/v1/company-news', params=params)

            return response.json()
        logging.info(f"{filen}, success.")
    except Exception as e:
        logging.error(f"{filen}, error {e}")


else:
    def get_news(ticker, dfrom,to):

        params = {
            'token': api_key,
            'symbol': ticker,
            'from': dfrom,
            'to': to
        }

        response = requests.get(url='https://finnhub.io/api/v1/company-news', params=params)

        return response.json()



Затем для каждого тикера мы находим новости за последний год

In [ ]:
if logs:
    try:
        logging.basicConfig(level=logging.INFO, filename="py_log.log",filemode="a",
                        format="%(asctime)s %(levelname)s %(message)s")
        tickers = ['AAPL', 'XOM', 'TSLA', 'WMT', 'PFE', 'NFLX', 'JPM', 'CAT']

        df_sub = pd.DataFrame()

        for ticker in tickers:
            news = get_news(ticker, dfrom='2025-05-01', to='2026-04-10')


            for new in news: 
                timestamp = new['datetime']
                date = datetime.fromtimestamp(timestamp, tz=timezone.utc).isoformat().replace('+00:00', 'Z') # перевод в формат iso 
                new['datetime'] = date

            df = pd.DataFrame(news)

            df_sub = pd.concat([df_sub,df], ignore_index=True)
        logging.info(f"{filen}, success.")
    except Exception as e:
        logging.error(f"{filen}, error {e}")

else:
    tickers = ['AAPL', 'XOM', 'TSLA', 'WMT', 'PFE', 'NFLX', 'JPM', 'CAT']

    df_sub = pd.DataFrame()

    for ticker in tickers:
        news = get_news(ticker, dfrom='2025-05-01', to='2026-04-10')


        for new in news: 
            timestamp = new['datetime']
            date = datetime.fromtimestamp(timestamp, tz=timezone.utc).isoformat().replace('+00:00', 'Z') # перевод в формат iso 
            new['datetime'] = date

        df = pd.DataFrame(news)

        df_sub = pd.concat([df_sub,df], ignore_index=True)

# 2026-03-23T13:31:06Z   

Убираем дубликаты и ненужные колонки 

In [ ]:
df_sub = df_sub.drop_duplicates()
df_sub = df_sub.drop(columns=['category','image'])
logging.info(f"{filen}, duplicates and columns deleted.")

Мерджим новости по id, объединяя компании 

In [ ]:
def custom_concat(x):
    return ';'.join(sorted(set(x)))

agg_df = df_sub.groupby('id').agg({
    'related': custom_concat,
}).reset_index()

dropped_df = df_sub.drop(columns=['related']).drop_duplicates(subset=['id'])
ыу
result_df = dropped_df.merge(agg_df, on='id')
logging.info(f"{filen}, dataset merged.")

Переназываем столбцы, чтобы мы все смержили 

In [ ]:
result_df = result_df.rename(columns={'datetime': 'webPublicationDate', 'summary':'fields.trailText', 'url':'webUrl'})

In [ ]:
result_df = result_df.rename(columns={'headline': 'fields.headline'})

In [ ]:
result_df = result_df.rename(columns={'related': 'stocks'})

In [ ]:
guardian_df = pd.read_csv('theguardian_raw_5.5k.csv')

main_df = pd.concat([guardian_df, result_df], ignore_index=True)

In [ ]:
main_df = main_df.drop(columns=['Unnamed: 0'])
main_df

Проставляем источник Api в датасет 

In [ ]:
main_df['source'] = main_df['source'].fillna('The Guardian')

In [ ]:
main_df['API_source'] = main_df['source'].apply(lambda source: 'FinHub' if source != 'The Guardian' else 'The Guardian')

In [ ]:
main_df['API_source'].value_counts()

Сохраение финального датасета

In [ ]:
main_df.to_csv('finhub+theguardian_raw_7.5k.csv')